In [1]:
%pip install -U anthropic
%pip install -U brotli

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Tool Calling

The pattern is extremely important for allowing the LLM to access information which isn't baked within it's internal weights. This is extremely important when the agent must stay up to date with recent events or any information where the agent shouldn't be expected to know the information own it's own.

In [2]:
from anthropic import Anthropic
import os
from IPython.display import display_markdown

In [3]:
client = Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY")
)

In [11]:
"""
Our fake weather tool 
"""

def weather_tool(location):
    
    lower_norm = location.lower().strip()
    if lower_norm == "pennsylvania":
        return 25
    
    return 50


In [12]:
system = """
You are a question answering agent and must answer the users questions with up to date information. In order to do this you have a set of tools which you can choose from. If the tool will help you
achieve the correct answer than you should call the tool else you should answer the question. For each tool call return a json object with the function name and arguments within <tool_all></tool_call> XML tags as follows:
<tool_call>
{"name": <function-name>, "argument": <arg-dict>}
</tool_call>

Here are the available tools:
<tools> {
    "name": "weather_tool",
    "description": "Get the current weather in a given location. TThe location can be a city or a state.",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            }
        }
    }
}
</tools>

Respond only to there question and do not provide any additional information at all.
"""

user = [{'role': 'user', 'content': 'what is the weather today in Pennsylvania'}]

r = client.messages.create(
    max_tokens=500,
    system=system,
    messages=user,
    model='claude-haiku-5-5'
)

resp = "".join(b.text for b in r.content if b.type=='text')

## Model Output

In [13]:
display_markdown(resp, raw=True)

<tool_call>
{"name": "weather_tool", "argument": {"location": "Pennsylvania"}}
</tool_call>

In [14]:
import re
import json
t = "<tool_call> this is the text </tool_call>"
result = re.search(r'<tool_call>(.*?)</tool_call>', resp, re.DOTALL)

tool_res = json.loads(result.group(1).strip())

temp = None
if tool_res['name'] == 'weather_tool':
    
    temp = weather_tool(tool_res['argument']['location'])

if temp is not None:
    user.append({'role': 'assistant', 'content': resp})
    user.append({'role': 'user', 'content': f'Observation: {temp}'})
    
    r = client.messages.create(
        max_tokens=500,
        system=system,
        messages=user,
        model = 'claude-haiku-5-5'
    )
    resp = "".join(b.text for b in r.content if b.type=='text')
    print(resp)

The weather in Pennsylvania today is 25 degrees.


## Next Steps

Above was an extremely simple version of tool calling which is extremely inefficient since we had to construct the system prompt by hand in order to include the needed tools. In a production based system there will be multiple tools which the LLM must choose from and it becomes very inefficient to update the prompt for every additional rule. Because of this all agentic frameworks extract the needed information from the actual functions and build the prompt using that information.

In [16]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.patterns import ToolAgent, tool

### Hacker News Tool

This is a simple tool which showcases how the tool calling actually works for real tools

In [17]:
import json
import requests

def fetch_top_hacker_news_stories(top_n: int):
    """
    Fetch the top stories from Hacker News.

    This function retrieves the top `top_n` stories from Hacker News using the Hacker News API. 
    Each story contains the title, URL, score, author, and time of submission. The data is fetched 
    from the official Firebase Hacker News API, which returns story details in JSON format.

    Args:
        top_n (int): The number of top stories to retrieve.
    """
    top_stories_url = 'https://hacker-news.firebaseio.com/v0/topstories.json'
    
    try:
        response = requests.get(top_stories_url)
        response.raise_for_status()  # Check for HTTP errors
        
        # Get the top story IDs
        top_story_ids = response.json()[:top_n]
        
        top_stories = []
        
        # For each story ID, fetch the story details
        for story_id in top_story_ids:
            story_url = f'https://hacker-news.firebaseio.com/v0/item/{story_id}.json'
            story_response = requests.get(story_url)
            story_response.raise_for_status()  # Check for HTTP errors
            story_data = story_response.json()
            
            # Append the story title and URL (or other relevant info) to the list
            top_stories.append({
                'title': story_data.get('title', 'No title'),
                'url': story_data.get('url', 'No URL available'),
            })
        
        return json.dumps(top_stories)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return []

In [18]:
json.loads(fetch_top_hacker_news_stories(top_n=5))

[{'title': 'Type Server Protocol',
  'url': 'https://github.com/microsoft/pylance-release/blob/main/docs/tsp/type-server-protocol.md'},
 {'title': "Sebastian Raschka's from-scratch video course on building reasoning models",
  'url': 'https://magazine.sebastianraschka.com/p/building-reasoning-models'},
 {'title': 'CSS-Tricks has a new owner',
  'url': 'https://css-tricks.com/css-tricks-has-a-new-owner/'},
 {'title': 'Oldest GUI application that still runs on Windows 11',
  'url': 'https://blog.enlyth.com/posts/oldest-windows-gui-app-win-11/'},
 {'title': 'Nietzschian Debugger',
  'url': 'https://github.com/DilawarShafiq/nietzschian-debugger'}]

In [19]:
hn_tool = tool(fetch_top_hacker_news_stories)

In [20]:
json.loads(hn_tool.fn_signature)

{'name': 'fetch_top_hacker_news_stories',
 'description': '\nFetch the top stories from Hacker News.\n\nThis function retrieves the top `top_n` stories from Hacker News using the Hacker News API. \nEach story contains the title, URL, score, author, and time of submission. The data is fetched \nfrom the official Firebase Hacker News API, which returns story details in JSON format.\n\nArgs:\n    top_n (int): The number of top stories to retrieve.\n',
 'parameters': {'properties': {'top_n': {'type': 'int'}}}}

In [21]:
tool_agent = ToolAgent(tools=[hn_tool], client=client, model_name='claude-haiku-5-5')

In [22]:
output = tool_agent.run(user_prompt="Tell me your name")

### No Tool Output

This prompt didn't require any tool calling thus it shouldn't have called any tools and should directly answer the users query

In [23]:
print(f'Tools called: {len(output.tools_used)}')
print(f'Total Input Tokens: {output.input_tokens}')
print(f'Total Output Tokens: {output.output_tokens}')
print(f'Total Latency: {output.latency}')

Tools called: 0
Total Input Tokens: 439
Total Output Tokens: 39
Total Latency: 0.8642772000748664


In [25]:
display_markdown(output.resp, raw=True)

I don't have a personal name. I'm Claude, an AI assistant made by Anthropic. How can I help you today?

In [28]:
output = tool_agent.run(user_prompt="Tell me the top 5 Hacker News stories right now", )

### Tool Calling output

This request requires the model to use the tool created. The model was trained was earlier than the current date which means that even if it for some reason had the top 5 highest ranked hacker news stories within it weights it wouldn't match the current rankings which is why the only way for the model to answer accurately is by calling the tool. It's important to understand that just because the tool is available it will not stop all hallucinations because calling the tool requires the model to recognise that it's unable to answer the question using it's own knowledge. I would assume smaller models not optimized for tool calling will often hallicinate producing answer which they thought where correct even though the only way to find the correct answer was through gathering information from a tool

In [29]:
print(f'Tools called: {len(output.tools_used)}')
print(f'Total Input Tokens: {output.input_tokens}')
print(f'Total Output Tokens: {output.output_tokens}')
print(f'Total Latency: {output.latency}')

Tools called: 1
Total Input Tokens: 1248
Total Output Tokens: 329
Total Latency: 2.341521499911323


In [30]:
display_markdown(output.resp, raw=True)

Here are the top 5 Hacker News stories right now:

1. **Type Server Protocol**
   https://github.com/microsoft/pylance-release/blob/main/docs/tsp/type-server-protocol.md

2. **Sebastian Raschka's from-scratch video course on building reasoning models**
   https://magazine.sebastianraschka.com/p/building-reasoning-models

3. **CSS-Tricks has a new owner**
   https://css-tricks.com/css-tricks-has-a-new-owner/

4. **Oldest GUI application that still runs on Windows 11**
   https://blog.enlyth.com/posts/oldest-windows-gui-app-win-11/

5. **Nietzschian Debugger**
   https://github.com/DilawarShafiq/nietzschian-debugger